<a href="https://colab.research.google.com/github/ultimatepin/card_recognizer/blob/main/notebooks/01_clip_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

colab is a cloud computer

! means a normal terminal command

## Clones the repo


In [9]:
!git clone https://github.com/ultimatepin/card_recognizer.git

Cloning into 'card_recognizer'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 43 (delta 9), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 3.48 MiB | 7.80 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [10]:
%cd card_recognizer

/content/card_recognizer/card_recognizer/card_recognizer


In [11]:
!ls

cards  notebooks  query.jpg  README.md


In [12]:
!pip -q install transformers pillow

## Import libraries

In [13]:
import os
import torch

from PIL import Image
from transformers import AutoProcessor, CLIPVisionModelWithProjection

In [14]:
MODEL_NAME = "openai/clip-vit-base-patch32"

# processor converts image to processable format
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# transformer and outputs embedding
model = CLIPVisionModelWithProjection.from_pretrained(MODEL_NAME)

# eval mode not train mode
model.eval()

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0

CLIPVisionModelWithProjection(
  (vision_model): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (position_embedding): Embedding(50, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)


## Turn image into embedding

In [15]:
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    # convert to pytorch tensor(pixel_values) to input model
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # not training the model, faster calc
    with torch.inference_mode():
        # pixel_values -> bunch of outputs
        output = model(**inputs)

    # embedding extraction
    embedding = output.image_embeds

    # Normalize the vector
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)

    return embedding

`get_embedding()` returns a single embedding vector corresponding to `image_path`.

Cosine similarity of two images is calculated using dot product/norm, higher cos value means higher similarity, meaning the angle between vectors determines similarity.

## Create embeddings from card library

In [16]:
card_folder = "cards"

card_embeddings = {}

for filename in os.listdir(card_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(card_folder, filename)

        card_embeddings[filename] = get_embedding(path)

print(f"Loaded {len(card_embeddings)} cards.")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Loaded 12 cards.


## Recognize `query.jpg`

In [17]:
query_embedding = get_embedding("query.jpg")

results = []

for card_name, card_embedding in card_embeddings.items():

    # .item() returns a float instead of a torch
    similarity = torch.sum(
        query_embedding * card_embedding
    ).item()

    results.append((card_name, similarity))

results.sort(
    key=lambda x: x[1],
    reverse=True
)

for card_name, score in results:
    print(f"{card_name:25} {score:.4f}")

OGN-249.png               0.9059
OGN-253.png               0.7982
OGN-265.png               0.7903
OGN-247.png               0.7767
OGN-263.png               0.7655
OGN-251.png               0.7475
OGN-269.png               0.7403
OGN-259.png               0.7377
OGN-257.png               0.7216
OGN-261.png               0.7059
OGN-255.png               0.6958
OGN-267.png               0.6660


Working!

Principle: image -> CLIP(pixel_value -> embedding) -> cosine similarities -> winner

##